## Business Analytics e Machine Learning Para Projetos de Data Science

### Data Warehousing Analytics Para Análise Geoespacial com Python e DuckDB

### Sobre o Projeto

Este notebook realiza análise geoespacial de dados de edifícios utilizando o dataset Google-Microsoft Open Buildings, que contém informações de mais de 2,5 bilhões de edifícios mapeados por inteligência artificial.

<strong>Áreas de Análise:</strong>

- Ceará, Brasil - Análise detalhada com aproximadamente 5-6 milhões de edifícios
- Uruguai - Fluxo completo de análsie com aproximadamente 3 milhões de edifícios

<strong> Tecnologias:</strong>

- DuckDB (banco de dados analítico)
- Extensão Spatial (operações geoespaciais)
- GeoParquet (formato de dados otimizado)

## 1. Configuração do Ambiente

In [53]:
# Instalação do pacotes
# !pip install -q duckdb geopandas pyarrow watermak

# Executar o arquivo requirements.txt
%pip install -r requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.


In [54]:
# Imports
import duckdb
import geopandas as gpd
import pandas as pd
import json
import os
from pathlib import Path

# Configurações de display
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [55]:
%reload_ext watermark
%watermark -a "Davi - Software Engineer" -v -m

Author: Davi - Software Engineer

Python implementation: CPython
Python version       : 3.13.11
IPython version      : 9.9.0

Compiler    : GCC 14.3.0
OS          : Linux
Release     : 6.14.0-37-generic
Machine     : x86_64
Processor   : x86_64
CPU cores   : 12
Architecture: 64bit



## 2. Inicialização do DuckDB

Utilizamos um banco de dados persistente para evitar recarregar os dados a cada execução. Isso reduz drasticamente o tempo de processamento após a primeira carga.

In [58]:
# Configuração do Banco de Dados
DB_PATH = "geospatial_analytics.duckdb"

# Conecta ao banco (cria se não existir)
con = duckdb.connect(database=DB_PATH)

# Configurações de performance
con.execute("SET memory_limit = '8gb';")
con.execute("SET threads = 4;")

print(f"Banco conectado: {DB_PATH}")

Banco conectado: geospatial_analytics.duckdb


In [57]:
# Instalação e carregamento das extensões
con.install_extension('httpfs')
con.load_extension('httpfs')
con.install_extension('spatial')
con.load_extension('spatial')

print("Extensões carregadas: httpfs, spatial")

Extensões carregadas: httpfs, spatial


In [ ]:
# Verifica tabelas existentes no banco
tables = con.execute("SHOW TABLES").fetchall()
table_names = [t[0] for t in tables]

print(f"Tabelas existentes: {table_names if table_names else 'Nenhuma (primeira execução)'}")

# Descomente a linha abaixo para realizar a limpeza - linha pode ser comentada depois
# con.execute("DROP TABLE IF EXISTS montevidoe_microsoft;")

Tabelas existentes: ['aoi_montevideo', 'montevideo_buildings_clipped', 'uruguai_buildings']


## 3. Fonte de Dados

O dataset Google-Microsoft Open Buildings está disponível no Source Cooperative em formato GeoParquet, particionado por país e grade S2.

<strong>Características do Dataset</strong>

- Cobertura Global (2.5B+ edifícios)
- Formato: GeoParquet
- Particionamento: Por país (ISO) e grade S2
- Fonte: Detecção por ML (Google + Microsoft)

In [59]:
# Configuração da fonte de dados
PREFIX = "s3://us-west-2.opendata.source.coop/vida/google-microsoft-open-buildings/geoparquet"

print(f"Fonte de Dados: {PREFIX}")

Fonte de Dados: s3://us-west-2.opendata.source.coop/vida/google-microsoft-open-buildings/geoparquet


---

## PARTE 1: Análise Completa do Uruguai

Iniciamos com o Uruguai por ser um dataset menor (aproximadamente 3 milhões de edifícios), permitindo demonstrar o fluxo completo de análise de forma rápida.

## 4. Carregamento dos Dados do Uruguai

In [62]:
%%time

# Carrega dados do Uruguai
COUNTRY_ISO_URY = "URY"

# ATUALIZA a lista de tabelas existentes
table_names  = [t[0] for t in con.execute("SHOW TABLES").fetchall()]

if "uruguai_buildings" not in table_names:
    print("Carregando dados do Uruguai do S3...")

    con.execute(f"""
        CREATE TABLE uruguai_buildings AS
        SELECT
            boundary_id,
            bf_source,
            confidence,
            area_in_meters,
            s2_id,
            geometry,
            geohash,
            bbox
        FROM parquet_scan('{PREFIX}/by_country_s2/country_iso={COUNTRY_ISO_URY}/*.parquet')    
    """)
    print("Dados do Uruguai carregados e persistidos!")
else:
    print("Tabela 'uruguai_buildings' encontrada em cache.")

Carregando dados do Uruguai do S3...
Dados do Uruguai carregados e persistidos!
CPU times: user 13.7 s, sys: 2.48 s, total: 16.2 s
Wall time: 1min 13s


## 5. Exploração dos Dados - Uruguai

In [63]:
%%time

con.query('DESCRIBE uruguai_buildings')

CPU times: user 2.35 ms, sys: 38 μs, total: 2.39 ms
Wall time: 1.43 ms


┌────────────────┬────────────────────────────────────────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name   │                        column_type                         │  null   │   key   │ default │  extra  │
│    varchar     │                          varchar                           │ varchar │ varchar │ varchar │ varchar │
├────────────────┼────────────────────────────────────────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ boundary_id    │ BIGINT                                                     │ YES     │ NULL    │ NULL    │ NULL    │
│ bf_source      │ VARCHAR                                                    │ YES     │ NULL    │ NULL    │ NULL    │
│ confidence     │ DOUBLE                                                     │ YES     │ NULL    │ NULL    │ NULL    │
│ area_in_meters │ DOUBLE                                                     │ YES     │ NULL    │ NULL    │ NULL    │
│ s2_id          │ BIGINT               

In [64]:
%%time

# Contagem total de edifícios
con.query('SELECT COUNT(*) AS total_edificios FROM uruguai_buildings')

CPU times: user 1.34 ms, sys: 46 μs, total: 1.39 ms
Wall time: 784 μs


┌─────────────────┐
│ total_edificios │
│      int64      │
├─────────────────┤
│         3100386 │
└─────────────────┘

In [65]:
%%time

# Distribuição por fonte de dados
con.query("""
    SELECT
        bf_source AS fonte_dados,
        COUNT(*) AS quantidade,
        ROUND(COUNT(*) * 100.00 / SUM(COUNT(*)) OVER(), 2) AS percentual
    FROM uruguai_buildings
    GROUP BY bf_source
    ORDER BY quantidade DESC
    """)

CPU times: user 2.26 ms, sys: 28 μs, total: 2.29 ms
Wall time: 1.48 ms


┌─────────────┬────────────┬────────────┐
│ fonte_dados │ quantidade │ percentual │
│   varchar   │   int64    │   double   │
├─────────────┼────────────┼────────────┤
│ google      │    3026859 │      97.63 │
│ microsoft   │      73527 │       2.37 │
└─────────────┴────────────┴────────────┘

In [66]:
%%time

# Amostra de Dados
con.query('SELECT * FROM uruguai_buildings LIMIT 10')

CPU times: user 557 μs, sys: 957 μs, total: 1.51 ms
Wall time: 913 μs


┌─────────────┬───────────┬────────────┬────────────────┬──────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ boundary_id │ bf_source │ confidence │ area_in_meters │        s2_id         │                                                                                                                                                                            geometry                                                                                                                                                                            │ geohas

## 6. Análise Estatística por Partição S2 - Urauguai

As partições S2 são células de uma grade hierárquica que cobre toda a superfície terrestre. Analisa por S2 nos permite entender a distribuição geográfica dos edifícios.

In [71]:
%%time

# Contagem de edifícios por partição S2
con.query("""
    SELECT
        s2_id,
        COUNT(*) AS total_edificios,
        ROUND(AVG(area_in_meters), 2) AS area_media_m2,
        ROUND(AVG(confidence), 4) AS confianca_media
    FROM uruguai_buildings
    GROUP BY s2_id
    ORDER BY total_edificios DESC
    """)

CPU times: user 1.56 ms, sys: 957 μs, total: 2.52 ms
Wall time: 1.28 ms


┌──────────────────────┬─────────────────┬───────────────┬─────────────────┐
│        s2_id         │ total_edificios │ area_media_m2 │ confianca_media │
│        int64         │      int64      │    double     │     double      │
├──────────────────────┼─────────────────┼───────────────┼─────────────────┤
│ -7782220156096217088 │         3100386 │         79.01 │            0.78 │
└──────────────────────┴─────────────────┴───────────────┴─────────────────┘

In [72]:
%%time

# Média de edifícios partições S2
con.query("""
    SELECT ROUND(AVG(buildings_count), 0) AS
    media_edificios_por_particao
    FROM (
        SELECT s2_id, COUNT(*) AS buildings_count
        FROM uruguai_buildings
        GROUP BY s2_id   
    )
""")

CPU times: user 895 μs, sys: 995 μs, total: 1.89 ms
Wall time: 1.03 ms


┌──────────────────────────────┐
│ media_edificios_por_particao │
│            double            │
├──────────────────────────────┤
│                    3100386.0 │
└──────────────────────────────┘

## 7. Estatísticas Descritivas - Uruguai

In [73]:
%%time

# Estatísticas completas por fonte de dados
con.query("""
    SELECT
        bf_source AS fonte,
        COUNT(*) AS total,
        ROUND(MIN(area_in_meters), 2) AS area_min_m2,
        ROUND(AVG(area_in_meters), 2) AS area_media_m2,
        ROUND(MEDIAN(area_in_meters), 2) AS area_mediana_m2,
        ROUND(MAX(area_in_meters), 2) AS area_max_m2,
        ROUND(STDDEV(area_in_meters), 2) AS desvio_padrao,
        ROUND(AVG(confidence), 4) AS confianca_media
    FROM uruguai_buildings
    GROUP BY bf_source
""")

CPU times: user 2.08 ms, sys: 1.02 ms, total: 3.1 ms
Wall time: 1.74 ms


┌───────────┬─────────┬─────────────┬───────────────┬─────────────────┬─────────────┬───────────────┬─────────────────┐
│   fonte   │  total  │ area_min_m2 │ area_media_m2 │ area_mediana_m2 │ area_max_m2 │ desvio_padrao │ confianca_media │
│  varchar  │  int64  │   double    │    double     │     double      │   double    │    double     │     double      │
├───────────┼─────────┼─────────────┼───────────────┼─────────────────┼─────────────┼───────────────┼─────────────────┤
│ microsoft │   73527 │        6.84 │          62.3 │           27.12 │   168024.95 │       1038.67 │            NULL │
│ google    │ 3026859 │         2.5 │         79.42 │           51.34 │     35752.3 │        167.47 │            0.78 │
└───────────┴─────────┴─────────────┴───────────────┴─────────────────┴─────────────┴───────────────┴─────────────────┘

In [74]:
%%time

# Distribuição por faixa de confiança
con.query("""
    SELECT
        CASE
            WHEN confidence < 0.5 THEN '1. Baixa (<50%)'
            WHEN confidence < 0.7 THEN '2. Média (50-70%)'
            WHEN confidence < 0.9 THEN '3. Alta (70-90%)'
            ELSE '4. Muito Alta (>90%)' 
        END AS faixa_confianca,
        COUNT(*) AS quantidade,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentual
    FROM uruguai_buildings
    GROUP BY faixa_confianca
    ORDER BY faixa_confianca
""")

CPU times: user 2.45 ms, sys: 847 μs, total: 3.3 ms
Wall time: 1.88 ms


┌──────────────────────┬────────────┬────────────┐
│   faixa_confianca    │ quantidade │ percentual │
│       varchar        │   int64    │   double   │
├──────────────────────┼────────────┼────────────┤
│ 2. Média (50-70%)    │     506375 │      16.33 │
│ 3. Alta (70-90%)     │    2416537 │      77.94 │
│ 4. Muito Alta (>90%) │     177474 │       5.72 │
└──────────────────────┴────────────┴────────────┘

In [75]:
%%time

# Distribuição por faixa de área
con.query("""
    SELECT
        CASE
            WHEN area_in_meters < 50 THEN '1. Pequeno(<50m²)'
            WHEN area_in_meters < 100 THEN '2. Médio(50-100m²)'
            WHEN area_in_meters < 200 THEN '3. Grande(100-200m²)'
            WHEN area_in_meters < 500 THEN '4. Muito Grande(200-500m²)'
            ELSE '5. Excepcional (>500m²)'
        END AS faixa_area,
        COUNT(*) AS quantidade,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentual
    FROM uruguai_buildings
    GROUP BY faixa_area
    ORDER BY faixa_area
 """)

CPU times: user 2.36 ms, sys: 150 μs, total: 2.51 ms
Wall time: 1.28 ms


┌────────────────────────────┬────────────┬────────────┐
│         faixa_area         │ quantidade │ percentual │
│          varchar           │   int64    │   double   │
├────────────────────────────┼────────────┼────────────┤
│ 1. Pequeno(<50m²)          │    1540350 │      49.68 │
│ 2. Médio(50-100m²)         │     880011 │      28.38 │
│ 3. Grande(100-200m²)       │     520268 │      16.78 │
│ 4. Muito Grande(200-500m²) │     129998 │       4.19 │
│ 5. Excepcional (>500m²)    │      29759 │       0.96 │
└────────────────────────────┴────────────┴────────────┘

## 8. Área de Interesse - Montevidéu

In [92]:
# Bounding Box de Montevidéu (capital Uruguai)

MONTEVIDEO_BBOX = {
    "min_lon": -56.35,
    "max_lon": -56.05,
    "min_lat": -34.95,
    "max_lat": -34.80
}

# Criar GeoJSON da área de interesse
montevideo_geojson = {
    "type": "FeatureCollection",
    "features": [{
        "type": "Feature",
        "properties": {"name": "Montevidéu", "country": "Uruguai"},
        "geometry": {
            "type": "Polygon",
            "coordinates": [[
                [MONTEVIDEO_BBOX["min_lon"], MONTEVIDEO_BBOX["min_lat"]],
                [MONTEVIDEO_BBOX["max_lon"], MONTEVIDEO_BBOX["min_lat"]],
                [MONTEVIDEO_BBOX["max_lon"], MONTEVIDEO_BBOX["max_lat"]],
                [MONTEVIDEO_BBOX["min_lon"], MONTEVIDEO_BBOX["max_lat"]],
                [MONTEVIDEO_BBOX["min_lon"], MONTEVIDEO_BBOX["min_lat"]]
            ]]
        }
    }]
}

with open("data/montevideo_aoi.geojson", "w") as f:
    json.dump(montevideo_geojson, f)

print(f"AOI Montevidéu criado: {MONTEVIDEO_BBOX}")

AOI Montevidéu criado: {'min_lon': -56.35, 'max_lon': -56.05, 'min_lat': -34.95, 'max_lat': -34.8}


In [93]:
%%time

# Carrega a área de interesse no DuckDB
con.execute("DROP TABLE IF EXISTS aoi_montevideo")
con.execute("CREATE TABLE aoi_montevideo AS SELECT * FROM ST_Read('data/montevideo_aoi.geojson')")

con.query("SELECT * FROM aoi_montevideo")

CPU times: user 5.7 ms, sys: 4.18 ms, total: 9.88 ms
Wall time: 10.7 ms


┌────────────┬─────────┬─────────────────────────────────────────────────────────────────────────────────────┐
│    name    │ country │                                        geom                                         │
│  varchar   │ varchar │                                      geometry                                       │
├────────────┼─────────┼─────────────────────────────────────────────────────────────────────────────────────┤
│ Montevidéu │ Uruguai │ POLYGON ((-56.35 -34.95, -56.05 -34.95, -56.05 -34.8, -56.35 -34.8, -56.35 -34.95)) │
└────────────┴─────────┴─────────────────────────────────────────────────────────────────────────────────────┘

## 9. Clipping Espacial - Montevidéu

O <strong>clipping</strong> é uma operação que recorta os dados geoespaciais para uma área de interesse específica.

In [78]:
%%time

# Realiza o clipping dos edifícios para Montevidéu
con.execute("DROP TABLE IF EXISTS montevideo_buildings_clipped")

con.execute("""
    CREATE TABLE montevideo_buildings_clipped AS
    SELECT
        b.boundary_id,
        b.bf_source,
        b.confidence,
        b.area_in_meters,
        ST_Intersection(b.geometry, a.geom) AS geom
    FROM uruguai_buildings b, aoi_montevideo a
    WHERE ST_Intersects(b.geometry, a.geom)
""")

con.query("SELECT COUNT(*) AS total_montevideo FROM montevideo_buildings_clipped")

CPU times: user 13 s, sys: 186 ms, total: 13.2 s
Wall time: 4.01 s


┌──────────────────┐
│ total_montevideo │
│      int64       │
├──────────────────┤
│           584929 │
└──────────────────┘

In [80]:
%%time

# Estatísticas de Montevidéu
con.query("""
    SELECT 
        bf_source AS fonte,
        COUNT(*) AS total,
        ROUND(AVG(area_in_meters), 2) AS area_media_m2,
        ROUND(AVG(confidence), 4) AS confianca_media
    FROM montevideo_buildings_clipped
    GROUP BY bf_source
""")

CPU times: user 2.26 ms, sys: 0 ns, total: 2.26 ms
Wall time: 1.16 ms


┌───────────┬────────┬───────────────┬─────────────────┐
│   fonte   │ total  │ area_media_m2 │ confianca_media │
│  varchar  │ int64  │    double     │     double      │
├───────────┼────────┼───────────────┼─────────────────┤
│ google    │ 579340 │         89.07 │          0.7672 │
│ microsoft │   5589 │         60.98 │            NULL │
└───────────┴────────┴───────────────┴─────────────────┘

In [82]:
%%time

# Verificação de valores de confidence por fonte
con.query("""
    SELECT
        bf_source,
        COUNT(*) AS total,
        COUNT(confidence) AS com_confidence,
        COUNT(*) - COUNT(confidence) AS sem_confidence
    FROM montevideo_buildings_clipped
    GROUP BY bf_source
""")

CPU times: user 2.31 ms, sys: 27 μs, total: 2.33 ms
Wall time: 1.18 ms


┌───────────┬────────┬────────────────┬────────────────┐
│ bf_source │ total  │ com_confidence │ sem_confidence │
│  varchar  │ int64  │     int64      │     int64      │
├───────────┼────────┼────────────────┼────────────────┤
│ google    │ 579340 │         579340 │              0 │
│ microsoft │   5589 │              0 │           5589 │
└───────────┴────────┴────────────────┴────────────────┘

## 10. Comparação Google vs Microsoft - Uruguai

Identificamos edifícios dectados exclusivamente por cada fonte de dados.

In [83]:
%%time

# Separa edifícios por fonte em Montevidéu
con.execute("DROP TABLE IF EXISTS montevideo_google")
con.execute("DROP TABLE IF EXISTS montevideo_microsoft")

con.execute("""
    CREATE TABLE montevideo_google AS
    SELECT * FROM montevideo_buildings_clipped WHERE bf_source = 'google'
""")

con.execute("""CREATE TABLE montevideo_microsoft AS
    SELECT * FROM montevideo_buildings_clipped WHERE bf_source = 'microsoft'
""")

google_count = con.execute("SELECT COUNT(*) FROM montevideo_google").fetchone()[0]
microsoft_count = con.execute("SELECT COUNT(*) FROM montevideo_microsoft").fetchone()[0]

print(f"Edifícios Google: {google_count:,}")
print(f"Edifícios Microsoft: {microsoft_count:,}")

Edifícios Google: 579,340
Edifícios Microsoft: 5,589
CPU times: user 981 ms, sys: 209 ms, total: 1.19 s
Wall time: 856 ms


In [84]:
%%time

# Edifícios Microsoft exclusivos (não detectados pelo Google)
con.execute("DROP TABLE IF EXISTS montevideo_microsoft_exclusivos")

con.execute("""
    CREATE TABLE montevideo_microsoft_exclusivos AS
    SELECT m.*
    FROM montevideo_microsoft m
    WHERE NOT EXISTS (
        SELECT 1
        FROM montevideo_google g
        WHERE ST_Intersects(m.geom, g.geom)
    )
""")

exclusivos = con.execute("SELECT COUNT(*) FROM montevideo_microsoft_exclusivos").fetchone()[0]
print(f"Edifícios exclusivos Microsoft: {exclusivos:,}")
print(f"  (Detectados apenas pela Microsoft, não pelo Google)")

Edifícios exclusivos Microsoft: 5,589
  (Detectados apenas pela Microsoft, não pelo Google)
CPU times: user 770 ms, sys: 14.4 ms, total: 784 ms
Wall time: 399 ms


## 11. Exportação - Uruguai

In [88]:
%%time

# Exporta edifícios de Montevidéu para FlatGeobuf
output_file = "outputs/montevideo_buildings.fgb"

con.execute(f"""
    COPY (
        SELECT boundary_id, bf_source, confidence, area_in_meters, geom
        FROM montevideo_buildings_clipped
    ) TO '{output_file}' WITH (FORMAT GDAL, DRIVER 'FlatGeobuf')
""")

print(f"Exportado: {output_file}")

Exportado: outputs/montevideo_buildings.fgb
CPU times: user 3.26 s, sys: 371 ms, total: 3.63 s
Wall time: 3.63 s


In [89]:
%%time

# Exporta exclusivos Microsoft
output_file2 = "outputs/montevideo_microsoft_exclusivos.fgb"

con.execute(f"""
    COPY (
        SELECT boundary_id, confidence, area_in_meters, geom
        FROM montevideo_microsoft_exclusivos
    ) TO '{output_file2}' WITH (FORMAT GDAL, DRIVER 'FlatGeobuf')
""")

print(f"Exportado: {output_file2}")

Exportado: outputs/montevideo_microsoft_exclusivos.fgb
CPU times: user 119 ms, sys: 8.09 ms, total: 127 ms
Wall time: 64.2 ms


---
## PARTE 2 (Brasil): Análise do Estado do Ceará

O Cearé possui aproximadamente <strong>5-6 milhões de edifícios</strong>, sendo uma amostra representativa do Nordeste brasileiro. Utilizamos filtragem por bounding box para otimizar o carregamento.

## 12. Definição da Área - Ceará

In [91]:
# Bounding Box do Estado do Ceará
CEARA_BBOX = {
    "min_lon": -41.5,
    "max_lon": -37.2,
    "min_lat": -7.9,
    "max_lat": -2.8
}

# Criar GeoJSON
ceara_geojson = {
    "type": "FeatureCollection",
    "features": [{
        "type": "Feature",
        "properties": {"nome": "Ceará", "sigla": "CE", "country": "Brasil"},
        "geometry": {
            "type": "Polygon",
            "coordinates": [[
                [CEARA_BBOX["min_lon"], CEARA_BBOX["min_lat"]],
                [CEARA_BBOX["max_lon"], CEARA_BBOX["min_lat"]],
                [CEARA_BBOX["max_lon"], CEARA_BBOX["max_lat"]],
                [CEARA_BBOX["min_lat"], CEARA_BBOX["max_lat"]],
                [CEARA_BBOX["min_lon"], CEARA_BBOX["min_lat"]]
            ]]
        }
    }]
}

with open("data/ceara_bbox.geojson", "w") as f:
    json.dump(ceara_geojson, f)

print("AOI Ceará definida:")
print(f"  Longitude: {CEARA_BBOX['min_lon']}° a {CEARA_BBOX['max_lon']}°")
print(f"  Latitude: {CEARA_BBOX['min_lat']}° a {CEARA_BBOX['max_lat']}")

AOI Ceará definida:
  Longitude: -41.5° a -37.2°
  Latitude: -7.9° a -2.8


## 13. Carregamento dos Dados do Ceará

Aplicamos o filtro de bounding box <strong>durante o donwload</strong>, reduzindo significativamente o volume de dados transferidos e armazenados.

In [94]:
%%time

# Carrega dados do Ceará com filtro espacial
COUNTRY_ISO_BRA = "BRA"

if "ceara_buildings" not in table_names:
    print("  Carregando dados do Ceará do S3...")
    print("  Aplicando filtro por bounding box durante o download.")
    print("  Isso pode levar alguns minutos na primeira execução. \n")

    con.execute(f"""
        CREATE TABLE ceara_buildings AS 
        SELECT 
            boundary_id,
            bf_source,
            confidence,
            area_in_meters,
            s2_id,
            geometry,
            geohash,
            bbox
        FROM parquet_scan('{PREFIX}/by_country_s2/country_iso={COUNTRY_ISO_BRA}/*.parquet')
        WHERE
            bbox.xmin >= {CEARA_BBOX['min_lon']} AND
            bbox.xmax <= {CEARA_BBOX['max_lon']} AND
            bbox.ymin >= {CEARA_BBOX['min_lat']} AND
            bbox.ymax <= {CEARA_BBOX['max_lat']}
    """)

    print("Dados do Ceará carregados e persistidos!")
else:
    print("Tabela 'ceara_buildings' encontrada em cache.")

  Carregando dados do Ceará do S3...
  Aplicando filtro por bounding box durante o download.
  Isso pode levar alguns minutos na primeira execução. 

Dados do Ceará carregados e persistidos!
CPU times: user 42.9 s, sys: 8.57 s, total: 51.5 s
Wall time: 10min


## 14. Exploração dos Dados - Ceará

In [95]:
%%time

# Estrutura da tabela
con.query('DESCRIBE ceara_buildings')

CPU times: user 2.28 ms, sys: 0 ns, total: 2.28 ms
Wall time: 1.22 ms


┌────────────────┬────────────────────────────────────────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name   │                        column_type                         │  null   │   key   │ default │  extra  │
│    varchar     │                          varchar                           │ varchar │ varchar │ varchar │ varchar │
├────────────────┼────────────────────────────────────────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ boundary_id    │ BIGINT                                                     │ YES     │ NULL    │ NULL    │ NULL    │
│ bf_source      │ VARCHAR                                                    │ YES     │ NULL    │ NULL    │ NULL    │
│ confidence     │ DOUBLE                                                     │ YES     │ NULL    │ NULL    │ NULL    │
│ area_in_meters │ DOUBLE                                                     │ YES     │ NULL    │ NULL    │ NULL    │
│ s2_id          │ BIGINT               

In [96]:
%%time

# Contagem total
result = con.query('SELECT COUNT(*) AS total_edificios FROM ceara_buildings')
print(result)

┌─────────────────┐
│ total_edificios │
│      int64      │
├─────────────────┤
│         6430541 │
└─────────────────┘

CPU times: user 9.87 ms, sys: 1.24 ms, total: 11.1 ms
Wall time: 4.85 ms


In [97]:
%%time

# Distribuição por fonte de dados
con.query("""
    SELECT
        bf_source AS fonte_dados,
        COUNT(*) AS quantidade,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS
        percentual
    FROM ceara_buildings
    GROUP BY bf_source
    ORDER BY quantidade DESC
""")

CPU times: user 1.34 ms, sys: 1.93 ms, total: 3.27 ms
Wall time: 1.59 ms


┌─────────────┬────────────┬────────────┐
│ fonte_dados │ quantidade │ percentual │
│   varchar   │   int64    │   double   │
├─────────────┼────────────┼────────────┤
│ google      │    6332302 │      98.47 │
│ microsoft   │      98239 │       1.53 │
└─────────────┴────────────┴────────────┘

In [98]:
%%time

# Estatísiticas descritivas completas
con.query("""
    SELECT 
        bf_source AS fonte,
        COUNT(*) AS total,
        ROUND(MIN(area_in_meters), 2) AS area_min_m2,
        ROUND(AVG(area_in_meters), 2) AS area_media_m2,
        ROUND(MEDIAN(area_in_meters), 2) AS area_mediana_m2,
        ROUND(MAX(area_in_meters), 2) AS area_max_m2,
        ROUND(STDDEV(area_in_meters), 2) AS desvio_padrao,
        ROUND(MIN(confidence), 4) AS conf_min,
        ROUND(AVG(confidence), 4) AS conf_media,
        ROUND(MAX(confidence), 4) AS conf_max
    FROM ceara_buildings
    GROUP BY bf_source
""")

CPU times: user 3.15 ms, sys: 1.9 ms, total: 5.05 ms
Wall time: 2.99 ms


┌───────────┬─────────┬─────────────┬───────────────┬─────────────────┬─────────────┬───────────────┬──────────┬────────────┬──────────┐
│   fonte   │  total  │ area_min_m2 │ area_media_m2 │ area_mediana_m2 │ area_max_m2 │ desvio_padrao │ conf_min │ conf_media │ conf_max │
│  varchar  │  int64  │   double    │    double     │     double      │   double    │    double     │  double  │   double   │  double  │
├───────────┼─────────┼─────────────┼───────────────┼─────────────────┼─────────────┼───────────────┼──────────┼────────────┼──────────┤
│ microsoft │   98239 │       10.04 │         67.42 │           44.47 │    90247.37 │        308.16 │     NULL │       NULL │     NULL │
│ google    │ 6332302 │        2.51 │        106.33 │           75.91 │    45671.05 │        197.81 │     0.65 │     0.8052 │    0.984 │
└───────────┴─────────┴─────────────┴───────────────┴─────────────────┴─────────────┴───────────────┴──────────┴────────────┴──────────┘

In [99]:
%%time

# Distribuição por faixa de confiança
con.query("""
    SELECT
        CASE
            WHEN confidence < 0.5 THEN '1. Baixa (<50%)'
            WHEN confidence < 0.7 THEN '2. Média (50-70%)'
            WHEN confidence < 0.9 THEN '3. Alto (70-90%)'
            ELSE '4. Muito Alta (>90%)'
        END AS faixa_confianca,
        COUNT(*) AS quantidade,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS
        percentual
    FROM ceara_buildings
    GROUP BY faixa_confianca
    ORDER BY faixa_confianca
""")

CPU times: user 2.85 ms, sys: 20 μs, total: 2.87 ms
Wall time: 2.17 ms


┌──────────────────────┬────────────┬────────────┐
│   faixa_confianca    │ quantidade │ percentual │
│       varchar        │   int64    │   double   │
├──────────────────────┼────────────┼────────────┤
│ 2. Média (50-70%)    │     788933 │      12.27 │
│ 3. Alto (70-90%)     │    5021108 │      78.08 │
│ 4. Muito Alta (>90%) │     620500 │       9.65 │
└──────────────────────┴────────────┴────────────┘

In [100]:
%%time

# Análise por partição S2
con.query("""
    SELECT
        s2_id,
        COUNT(*) AS total_edificios,
        ROUND(AVG(area_in_meters), 2) AS area_media_m2,
        ROUND(AVG(confidence), 4) AS confianca_medias
    FROM ceara_buildings
    GROUP BY s2_id
    ORDER BY total_edificios DESC
""")

CPU times: user 2.67 ms, sys: 0 ns, total: 2.67 ms
Wall time: 1.37 ms


┌────────────────────┬─────────────────┬───────────────┬──────────────────┐
│       s2_id        │ total_edificios │ area_media_m2 │ confianca_medias │
│       int64        │      int64      │    double     │      double      │
├────────────────────┼─────────────────┼───────────────┼──────────────────┤
│ 557320453887098880 │         1614085 │        103.84 │           0.8082 │
│ 550565054446043136 │         1554674 │        103.24 │           0.8027 │
│ 559572253700784128 │         1538369 │        120.66 │           0.8056 │
│ 546061454818672640 │          592459 │         93.39 │           0.8041 │
│ 548313254632357888 │          511808 │         86.12 │           0.8016 │
│ 570831252769210368 │          505824 │        107.31 │           0.8076 │
│ 552816854259728384 │           81423 │        111.94 │           0.7985 │
│ 555068654073413632 │           31899 │        106.94 │           0.8176 │
└────────────────────┴─────────────────┴───────────────┴──────────────────┘

In [101]:
%%time

# Amostra de dados
con.query('SELECT * FROM ceara_buildings WHERE bf_source = \'google\' LIMIT 10')

CPU times: user 678 μs, sys: 1.74 ms, total: 2.42 ms
Wall time: 1.25 ms


┌─────────────┬───────────┬────────────┬────────────────┬────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ boundary_id │ bf_source │ confidence │ area_in_meters │       s2_id        │                                                                                                                                         geometry                                                                                                                                         │ geohash  │                                                       bbox                                                       │
│    int64    │  varchar

## 15. Análise Estatística - Ceará

In [102]:
%%time

# Estatísticas descritivas completas
con.query("""
    SELECT
        bf_source AS fonte,
        COUNT(*) AS total,
        ROUND(MIN(area_in_meters), 2) AS area_min_m2,
        ROUND(AVG(area_in_meters), 2) AS area_media_m2,
        ROUND(MEDIAN(area_in_meters), 2) AS area_mediana_m2,
        ROUND(MAX(area_in_meters), 2) AS area_max_m2,
        ROUND(STDDEV(area_in_meters), 2) AS desvio_padrao,
        ROUND(MIN(confidence), 4) AS conf_min,
        ROUND(AVG(confidence), 4) AS confi_media,
        ROUND(MAX(confidence), 4) AS conf_max
    FROM ceara_buildings
    GROUP BY bf_source
""")

CPU times: user 3.14 ms, sys: 896 μs, total: 4.03 ms
Wall time: 2.39 ms


┌───────────┬─────────┬─────────────┬───────────────┬─────────────────┬─────────────┬───────────────┬──────────┬─────────────┬──────────┐
│   fonte   │  total  │ area_min_m2 │ area_media_m2 │ area_mediana_m2 │ area_max_m2 │ desvio_padrao │ conf_min │ confi_media │ conf_max │
│  varchar  │  int64  │   double    │    double     │     double      │   double    │    double     │  double  │   double    │  double  │
├───────────┼─────────┼─────────────┼───────────────┼─────────────────┼─────────────┼───────────────┼──────────┼─────────────┼──────────┤
│ microsoft │   98239 │       10.04 │         67.42 │           44.47 │    90247.37 │        308.16 │     NULL │        NULL │     NULL │
│ google    │ 6332302 │        2.51 │        106.33 │           75.91 │    45671.05 │        197.81 │     0.65 │      0.8052 │    0.984 │
└───────────┴─────────┴─────────────┴───────────────┴─────────────────┴─────────────┴───────────────┴──────────┴─────────────┴──────────┘

In [103]:
%%time 

# Distribuição por faixa de confiança
con.query("""
    SELECT
        CASE
            WHEN confidence < 0.5 THEN '1. Baixa (<50%)'
            WHEN confidence < 0.9 THEN '2. Média (50-70%)'
            WHEN confidence < 0.5 THEN '3. Alta (70-90%)'
            ELSE '4. Muito Alta (>90%)'
        END AS faixa_confianca,
        COUNT(*) AS quantidade,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentual
    FROM ceara_buildings
    GROUP BY faixa_confianca
    ORDER BY faixa_confianca
""")

CPU times: user 2.28 ms, sys: 34 μs, total: 2.32 ms
Wall time: 1.35 ms


┌──────────────────────┬────────────┬────────────┐
│   faixa_confianca    │ quantidade │ percentual │
│       varchar        │   int64    │   double   │
├──────────────────────┼────────────┼────────────┤
│ 2. Média (50-70%)    │    5810041 │      90.35 │
│ 4. Muito Alta (>90%) │     620500 │       9.65 │
└──────────────────────┴────────────┴────────────┘

In [104]:
%%time

# Análise por partição S2
con.query("""
    SELECT
        s2_id,
        COUNT(*) AS total_edificios,
        ROUND(AVG(area_in_meters), 2) AS area_media_m2,
        ROUND(AVG(confidence), 4) AS confianca_media
    FROM ceara_buildings
    GROUP BY s2_id
    ORDER BY total_edificios DESC
""")

CPU times: user 1.73 ms, sys: 0 ns, total: 1.73 ms
Wall time: 925 μs


┌────────────────────┬─────────────────┬───────────────┬─────────────────┐
│       s2_id        │ total_edificios │ area_media_m2 │ confianca_media │
│       int64        │      int64      │    double     │     double      │
├────────────────────┼─────────────────┼───────────────┼─────────────────┤
│ 557320453887098880 │         1614085 │        103.84 │          0.8082 │
│ 550565054446043136 │         1554674 │        103.24 │          0.8027 │
│ 559572253700784128 │         1538369 │        120.66 │          0.8056 │
│ 546061454818672640 │          592459 │         93.39 │          0.8041 │
│ 548313254632357888 │          511808 │         86.12 │          0.8016 │
│ 570831252769210368 │          505824 │        107.31 │          0.8076 │
│ 552816854259728384 │           81423 │        111.94 │          0.7985 │
│ 555068654073413632 │           31899 │        106.94 │          0.8176 │
└────────────────────┴─────────────────┴───────────────┴─────────────────┘

## 16. Área de Interesse - Região Metropolitana de Fortaleza

In [106]:
# Bounding Box da Região Metropolitana de Fortaleza
FORTALEZA_BBOX = {
    "min_lon": -38.75,
    "max_lon": -38.35,
    "min_lat": -3.90,
    "max_lat": -3.65
}

# Criar geoJSON
fortaleza_geojson = {
    "type": "FeatureCollection",
    "features": [{
        "type": "Feature",
        "properties": {"nome": "Fortaleza", "state": "CE", "country": "Brasil"},
        "geometry": {
            "type": "Polygon",
            "coordinates": [[
                [FORTALEZA_BBOX["min_lon"], FORTALEZA_BBOX["min_lat"]],
                [FORTALEZA_BBOX["max_lon"], FORTALEZA_BBOX['min_lat']],
                [FORTALEZA_BBOX["max_lon"], FORTALEZA_BBOX["max_lat"]],
                [FORTALEZA_BBOX["min_lon"], FORTALEZA_BBOX["max_lat"]],
                [FORTALEZA_BBOX["min_lon"], FORTALEZA_BBOX["min_lat"]]
            ]]
        }
    }]
}

with open("data/fortaleza_aoi.geojson", "w") as f:
    json.dump(fortaleza_geojson, f)

print(f"AOI Fortaleza definida: {FORTALEZA_BBOX}")

AOI Fortaleza definida: {'min_lon': -38.75, 'max_lon': -38.35, 'min_lat': -3.9, 'max_lat': -3.65}


In [108]:
%%time

# Carrega a AOI no DuckDB
con.execute("DROP TABLE IF EXISTS aoi_fortaleza")
con.execute("CREATE TABLE aoi_fortaleza AS SELECT * FROM ST_Read('data/fortaleza_aoi.geojson')")

con.query("SELECT * FROM aoi_fortaleza")

CPU times: user 12.5 ms, sys: 2.96 ms, total: 15.4 ms
Wall time: 17.3 ms


┌───────────┬─────────┬─────────┬───────────────────────────────────────────────────────────────────────────────┐
│   nome    │  state  │ country │                                     geom                                      │
│  varchar  │ varchar │ varchar │                                   geometry                                    │
├───────────┼─────────┼─────────┼───────────────────────────────────────────────────────────────────────────────┤
│ Fortaleza │ CE      │ Brasil  │ POLYGON ((-38.75 -3.9, -38.35 -3.9, -38.35 -3.65, -38.75 -3.65, -38.75 -3.9)) │
└───────────┴─────────┴─────────┴───────────────────────────────────────────────────────────────────────────────┘

## 17. Clipping Espacial - Fortaleza

In [109]:
%%time

# Clipping dos edifícios para Fortaleza
con.execute("DROP TABLE IF EXISTS fortaleza_buildings_clipped")

con.execute("""
    CREATE TABLE fortaleza_buildings_clipped AS
    SELECT
        b.boundary_id,
        b.bf_source,
        b.confidence,
        b.area_in_meters,
        ST_Intersection(b.geometry, a.geom) AS geom
    FROM ceara_buildings b, aoi_fortaleza a
    WHERE ST_Intersects(b.geometry, a.geom)
""")

con.query("SELECT COUNT(*) AS total_fortaleza FROM fortaleza_buildings_clipped")

CPU times: user 37.8 s, sys: 607 ms, total: 38.4 s
Wall time: 10.4 s


┌─────────────────┐
│ total_fortaleza │
│      int64      │
├─────────────────┤
│         1014323 │
└─────────────────┘

In [42]:
%%time
# Estatísticas de Fortaleza
con.query("""
    SELECT
        bf_source AS fonte,
        COUNT(*) AS total,
        ROUND(AVG(area_in_meters), 2) AS area_media_m2,
        ROUND(AVG(confidence), 4) AS confianca_media
    FROM fortaleza_buildings_clipped
    GROUP BY bf_source
""")

CPU times: user 1.28 ms, sys: 0 ns, total: 1.28 ms
Wall time: 774 μs


┌───────────┬─────────┬───────────────┬─────────────────┐
│   fonte   │  total  │ area_media_m2 │ confianca_media │
│  varchar  │  int64  │    double     │     double      │
├───────────┼─────────┼───────────────┼─────────────────┤
│ google    │ 1010064 │        127.56 │          0.8026 │
│ microsoft │    4259 │        143.82 │            NULL │
└───────────┴─────────┴───────────────┴─────────────────┘

## 18. Comparação Google vc Microsoft - Fortaleza

In [110]:
%%time

# Separa por fonte
con.execute("DROP TABLE IF EXISTS fortaleza_google")
con.execute("DROP TABLE IF EXISTS fortaleza_microsoft")

con.execute("""
    CREATE TABLE fortaleza_google AS
    SELECT * FROM fortaleza_buildings_clipped WHERE bf_source = 'google'
""")

con.execute("""
    CREATE TABLE fortaleza_microsoft AS
    SELECT * FROM fortaleza_buildings_clipped WHERE bf_source = 'microsoft'
""")

google_count = con.execute("SELECT COUNT(*) FROM fortaleza_google").fetchone()[0]
microsoft_count = con.execute("SELECT COUNT(*) FROM fortaleza_microsoft").fetchone()[0]

print(f"Edíficios Google: {google_count:,}")
print(f"Edíficios Microsoft: {google_count:,}")

Edíficios Google: 1,010,064
Edíficios Microsoft: 1,010,064
CPU times: user 3.07 s, sys: 567 ms, total: 3.63 s
Wall time: 1.7 s


In [111]:
%%time

# Edifícios Microsoft exclusivos
con.execute("DROP TABLE IF EXISTS fortaleza_microsoft_exclusivos")

con.execute("""
    CREATE TABLE fortaleza_microsoft_exclusivos AS
    SELECT m.*
    FROM fortaleza_microsoft m
    WHERE NOT EXISTS (
        SELECT 1
        FROM fortaleza_google g
        WHERE ST_Intersects(m.geom, g.geom)
    )
""")

exclusivos = con.execute("SELECT COUNT(*) FROM fortaleza_microsoft_exclusivos").fetchone()[0]
print(f"Edifícios exclusivos Microsoft em Fortaleza: {exclusivos:,}")

Edifícios exclusivos Microsoft em Fortaleza: 4,259
CPU times: user 175 ms, sys: 7.09 ms, total: 183 ms
Wall time: 175 ms


## 19. Exportação - Ceará

In [112]:
%%time 

# Exportação Fortaleza completa
output_file = "outputs/fortaleza_buildings.fgb"

con.execute(f"""
    COPY (
        SELECT boundary_id, bf_source, confidence, area_in_meters, geom
        FROM fortaleza_buildings_clipped
    ) TO '{output_file}' WITH (FORMAT GDAL, DRIVER 'FlatGeobuf')
""")

print(f"Exportado: {output_file}")

Exportado: outputs/fortaleza_buildings.fgb
CPU times: user 2.69 s, sys: 335 ms, total: 3.03 s
Wall time: 3.03 s


In [47]:
%%time

# Exporta exclusivos Microsoft
output_file2 = "fortaleza_microsoft_exclusivos.fgb"

con.execute(f"""
    COPY (
        SELECT boundary_id, confidence, area_in_meters, geom
        FROM fortaleza_microsoft_exclusivos
    ) TO '{output_file2}' WITH (FORMAT GDAL, DRIVER 'FlatGeobuf')
""")

print(f"Exportados: {output_file2}")

Exportados: fortaleza_microsoft_exclusivos.fgb
CPU times: user 21.9 ms, sys: 2.82 ms, total: 24.7 ms
Wall time: 16.9 ms


---

## 20. Resumo e Métricas de Performance

In [113]:
# Resumo das tabelas criadas
print("=" * 70)
print("RESUMO DO PROJETO")
print("=" * 70)

tables_info = con.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
    ORDER BY table_name
""").fetchall()

print("\n Tabelas no banco de dados: \n")
for table in tables_info:
    table_name = table[0]
    count = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    print(f"  - {table_name}: {count:,} registros")

print("\n" + "=" * 70)

RESUMO DO PROJETO

 Tabelas no banco de dados: 

  - aoi_fortaleza: 1 registros
  - aoi_montevideo: 1 registros
  - ceara_buildings: 6,430,541 registros
  - fortaleza_buildings_clipped: 1,014,323 registros
  - fortaleza_google: 1,010,064 registros
  - fortaleza_microsoft: 4,259 registros
  - fortaleza_microsoft_exclusivos: 4,259 registros
  - montevideo_buildings_clipped: 584,929 registros
  - montevideo_google: 579,340 registros
  - montevideo_microsoft: 5,589 registros
  - montevideo_microsoft_exclusivos: 5,589 registros
  - uruguai_buildings: 3,100,386 registros



In [116]:
# Tamanho do banco de dados e arquivos exportados
print("\n ARMAZENAMENTO: \n")

# Banco de dados
if os.path.exists(DB_PATH):
    db_size = os.path.getsize(DB_PATH) / (1024 * 1024)
    print(f"   - Banco de dados ({DB_PATH}): {db_size:.2f} MB")

# Arquivos exportados
expost_file = [
    "outputs/montevideo_buildings.fgb",
    "outputs/montevideo_microsoft_exclusivos.fgb",
    "outputs/fortaleza_buildings.fgb",
    "outputs/fortaleza_microsoft_exclusivos.fgb"
]

print("\n Arquivos exportados: ")
for f in expost_file:
    if os.path.exists(f):
        size = os.path.getsize(f) / (1024 * 1024)
        print(f"  - {f}: {size:.2f}")


 ARMAZENAMENTO: 

   - Banco de dados (geospatial_analytics.duckdb): 1731.51 MB

 Arquivos exportados: 
  - outputs/montevideo_buildings.fgb: 135.86
  - outputs/montevideo_microsoft_exclusivos.fgb: 1.14
  - outputs/fortaleza_buildings.fgb: 238.37
  - outputs/fortaleza_microsoft_exclusivos.fgb: 0.88


In [117]:
# Comparativo de otimização
print("\n COMPARATIVO DE OTIMIZAÇÃO: \n")
print("  Abordagem tradicional (Brasil completo): ")
print("  - Registros: ~141 milhões")
print("  - Armazenamento: ~22 GB")
print("  - Tempo de carga: 20-40 minutos")
print("  - RAM necessária: 32+ GB")

ceara_count = con.execute("SELECT COUNT(*) FROM ceara_buildings").fetchone()[0]
db_sice_mb = os.path.getsize(DB_PATH) / (1024 * 1024) if os.path.exists(DB_PATH) else 0

print("\n  Abordagem otimizada (Ceará + Uruguai): ")
print(f"   - Registros Ceará: ~{ceara_count/1_000_000:.1f} milhões")
print(f"   - Armazenamento total: ~{db_sice_mb:.0f} MB")
print("   - Tempo de carga: 3-5 minutos (primeira vez)")
print("   - RAM necessária: 8-16 GB")
print(f"\n  Redução de armazenamento: ~{22000/db_sice_mb:.0f}x menor" if db_sice_mb > 0 else "")


 COMPARATIVO DE OTIMIZAÇÃO: 

  Abordagem tradicional (Brasil completo): 
  - Registros: ~141 milhões
  - Armazenamento: ~22 GB
  - Tempo de carga: 20-40 minutos
  - RAM necessária: 32+ GB

  Abordagem otimizada (Ceará + Uruguai): 
   - Registros Ceará: ~6.4 milhões
   - Armazenamento total: ~1732 MB
   - Tempo de carga: 3-5 minutos (primeira vez)
   - RAM necessária: 8-16 GB

  Redução de armazenamento: ~13x menor


In [118]:
# Fecha a conexão com o banco 
con.close()
print(f"\n Conexão fechada. Dados persistidos em: {DB_PATH}")
print("  Na próxima execução, os dados serão carregados instanteamente do cache.")


 Conexão fechada. Dados persistidos em: geospatial_analytics.duckdb
  Na próxima execução, os dados serão carregados instanteamente do cache.


In [119]:
%watermark -a "Davi - Software Engineer" --iversions

Author: Davi - Software Engineer

duckdb   : 1.4.3
geopandas: 1.1.2
json     : 2.0.9
pandas   : 2.3.3



# FIM 